# Phase 3 — Data Quality Validation with Great Expectations

Validates `silver.orders` using GX's Pandas execution engine — GX's Spark
engine calls Spark's `persist()`/`cache()` internally, which Databricks
Serverless compute doesn't support, so the table is collected to the
driver instead (small enough at order-item grain, ~100k rows). Four
required expectations — not-null `order_id`, not-null `customer_id`,
`order_amount` strictly positive, `order_date` within the dataset's known
range — plus a stretch referential-integrity check that every
`customer_id` exists in `bronze.customers`. The resulting HTML Data Docs are
rendered inline and also packaged into a zip on the Unity Catalog volume
for a properly-styled local view.

In [ ]:
%pip install -q great_expectations

In [ ]:
dbutils.library.restartPython()

In [ ]:
import datetime
import os

import great_expectations as gx
from great_expectations.checkpoint import UpdateDataDocsAction
from great_expectations.exceptions import DataContextError

## Set up a file-based Data Context

A file-based (rather than ephemeral) context is used so the generated
Data Docs are written to disk as a static HTML site. The project
directory is wiped before each run so re-executing the notebook within
the same cluster session starts clean instead of colliding with objects
a prior run already registered. A fresh context already comes with a
default `local_site` Data Docs site at `uncommitted/data_docs/local_site/`,
so no explicit site config is needed.

In [ ]:
import shutil

GX_PROJECT_ROOT = "/tmp/gx_project"
shutil.rmtree(GX_PROJECT_ROOT, ignore_errors=True)

context = gx.get_context(mode="file", project_root_dir=GX_PROJECT_ROOT)

## Connect to silver.orders as a pandas dataframe batch

In [ ]:
data_source = context.data_sources.add_pandas(name="silver_orders_datasource")
data_asset = data_source.add_dataframe_asset(name="silver_orders")
batch_definition = data_asset.add_batch_definition_whole_dataframe("silver_orders_batch")

silver_orders_df = spark.table("silver.orders").toPandas()
batch_parameters = {"dataframe": silver_orders_df}

## Define the Expectation Suite

The 4 required expectations. Adding the suite falls back to fetching the
existing one by name if this cell is re-run within the same session —
its expectations are then reset before re-adding them, rather than
appending duplicates onto whatever the suite already had.

In [ ]:
SILVER_ORDERS_SUITE_NAME = "silver_orders_suite"

try:
    suite = context.suites.add(gx.ExpectationSuite(name=SILVER_ORDERS_SUITE_NAME))
except DataContextError:
    suite = context.suites.get(SILVER_ORDERS_SUITE_NAME)
    suite.expectations = []

suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_amount", min_value=0, strict_min=True
    )
)
_ = suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_date",
        min_value=datetime.date(2016, 1, 1),
        max_value=datetime.date(2018, 12, 31),
    )
)

## Stretch expectation — referential integrity

Every `customer_id` in Silver orders must exist in `bronze.customers`.
GX has no built-in foreign-key expectation, so this is approximated by
checking set membership against the distinct customer_id list — fine at
this dataset's scale, but collecting the full distinct list to the
driver wouldn't hold up on a much larger customers table.

In [ ]:
known_customer_ids = [
    row["customer_id"]
    for row in spark.table("bronze.customers").select("customer_id").distinct().collect()
]

_ = suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="customer_id", value_set=known_customer_ids
    )
)

## Validate — build a Validation Definition and run a Checkpoint

The Checkpoint's `UpdateDataDocsAction` regenerates the HTML Data Docs
with the new Validation Results after it runs. Both the Validation
Definition and the Checkpoint fall back to fetching the existing object
by name on re-run within the same session, since GX's `.add()` isn't
idempotent.

In [ ]:
try:
    validation_definition = context.validation_definitions.add(
        gx.ValidationDefinition(
            name="silver_orders_validation",
            data=batch_definition,
            suite=suite,
        )
    )
except DataContextError:
    validation_definition = context.validation_definitions.get("silver_orders_validation")

try:
    checkpoint = context.checkpoints.add(
        gx.Checkpoint(
            name="silver_orders_checkpoint",
            validation_definitions=[validation_definition],
            actions=[UpdateDataDocsAction(name="update_all_data_docs")],
        )
    )
except DataContextError:
    checkpoint = context.checkpoints.get("silver_orders_checkpoint")

checkpoint_result = checkpoint.run(batch_parameters=batch_parameters)

## Pass/fail summary

In [ ]:
print(f"Checkpoint success: {checkpoint_result.success}\n")
print(checkpoint_result.describe())

## Data Docs

GX nests its project files a couple of levels inside `GX_PROJECT_ROOT`
(under a `great_expectations/` subfolder), so the Data Docs `index.html`
path is discovered with `glob` rather than hardcoded.

In [ ]:
import glob

data_docs_path = glob.glob(f"{GX_PROJECT_ROOT}/**/local_site/index.html", recursive=True)[0]

with open(data_docs_path, "r") as f:
    data_docs_html = f.read()

displayHTML(data_docs_html)

## Package Data Docs for download

The HTML site is zipped and copied to the Unity Catalog volume for
download — the volume is FUSE-mounted over cloud storage, so the zip is
built on local disk first (`zipfile` needs random-access seeks the mount
doesn't support) and then copied over with a plain sequential write.

In [ ]:
local_site_dir = os.path.dirname(data_docs_path)
zip_path = shutil.make_archive("/tmp/gx_data_docs", "zip", local_site_dir)
shutil.copyfile(zip_path, "/Volumes/de_capstone/default/olist_data/gx_data_docs.zip")

print(f"Copied {os.path.getsize(zip_path)} bytes to the volume")

## Return the result to the caller

Phase 4's Airflow DAG triggers this notebook via the Databricks Jobs API
and needs the checkpoint result to branch on — `dbutils.notebook.exit`
is how a triggered notebook run passes a value back to its caller.

In [ ]:
import json

dbutils.notebook.exit(json.dumps({"success": checkpoint_result.success}))